# 04 — XGBoost Baseline (Fraud Scoring)

Trains the supervised fraud-scoring model on time-window features.

**Evaluation design:** time-based 70/10/20 split (train / validation / test) — the model is always tested on *future* transactions, never on a random shuffle. Random splits leak information for per-customer time-series features and inflate metrics.

**Outputs:** `models/xgb_fraud_model.pkl`, `models/model_metrics.json`, PR-curve and feature-importance charts in `reports/`.

In [ ]:
# CELL 1 - Load features, sort chronologically (required!)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/time_window_features.csv")
df["TransactionDT"] = pd.to_datetime(df["TransactionDT"])

# Chronological order is REQUIRED for the time-based split below.
df = df.sort_values("TransactionDT").reset_index(drop=True)

os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)
print(f"Loaded: {df.shape}")

In [ ]:
# CELL 2 - velocity_risk fallback (same formula as notebook 03)
if "velocity_risk" not in df.columns:
    df["velocity_risk"] = (
        df["txn_count_1h"] / df["txn_count_24h"].clip(lower=1)
    ).clip(0, 1)

In [ ]:
# CELL 3 - Feature set (uses new features from fixed notebook 02)
features = [
    # velocity
    "txn_count_1h", "txn_count_24h", "txn_count_7d", "velocity_risk",
    # amount behavior vs prior baseline
    "avg_amt_1h", "avg_amt_24h", "avg_amt_7d", "max_amt_24h",
    "amount_dev_24h", "amount_zscore_24h",
    # time
    "hour", "day_of_week", "is_night_txn",
    # history depth
    "no_prior_24h",
]
missing = [f for f in features if f not in df.columns]
assert not missing, f"Missing columns: {missing} - re-run notebooks 01-03 first"

X = df[features]
y = df["isFraud"]

In [ ]:
# CELL 4 - TIME-BASED split (replaces random split)
# Random splits leak: a customer's future transactions land in train
# while their past sits in test, inflating metrics. Instead: train on
# the past, validate on the middle, test on the FUTURE.
n = len(df)
train_end, val_end = int(n * 0.70), int(n * 0.80)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val     = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test   = X.iloc[val_end:], y.iloc[val_end:]

for name, s in [("train", slice(0, train_end)),
                ("val", slice(train_end, val_end)),
                ("test", slice(val_end, n))]:
    part = df.iloc[s]
    print(f"{name:>5}: {len(part):>7,} rows | "
          f"{part['TransactionDT'].min():%Y-%m-%d} -> "
          f"{part['TransactionDT'].max():%Y-%m-%d} | "
          f"fraud rate {part['isFraud'].mean():.2%}")

In [ ]:
# CELL 5 - Class imbalance weight (computed once)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")
# NOTE: this intentionally inflates raw scores ~26x above the base rate.
# Scores are used for RANKING transactions; thresholds are chosen from
# the validation/test tradeoff table below, not read as calibrated
# probabilities.

In [ ]:
# CELL 6 - Train with early stopping on the validation window
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=2000,            # upper bound; early stopping picks the real number
    max_depth=6,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",          # PR-AUC: informative at 3.7% base rate
    early_stopping_rounds=50,
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"best iteration: {model.best_iteration}")

In [ ]:
# CELL 7 - Evaluate on the held-out FUTURE test window
from sklearn.metrics import roc_auc_score, average_precision_score

yt = y_test.to_numpy()
y_scores = model.predict_proba(X_test)[:, 1]

roc = roc_auc_score(yt, y_scores)
ap = average_precision_score(yt, y_scores)
base = yt.mean()

print(f"ROC-AUC:            {roc:.4f}")
print(f"PR-AUC (avg prec.): {ap:.4f}   (baseline rate = {base:.4f})")
print(f"lift over baseline: {ap / base:.1f}x")

In [ ]:
# CELL 8 - PR curve + threshold tradeoff (saved correctly)
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(yt, y_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(recall, precision)
axes[0].axhline(base, ls="--", c="gray", lw=1, label=f"baseline {base:.3f}")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve (test = future data)")
axes[0].legend()
axes[1].plot(thresholds, precision[:-1], label="Precision")
axes[1].plot(thresholds, recall[:-1], label="Recall")
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("Threshold Tradeoff"); axes[1].legend()
plt.tight_layout()
plt.savefig("../reports/xgb_baseline_pr_curve.png", dpi=150,
            bbox_inches="tight")   # BEFORE show()
plt.show()

In [ ]:
# CELL 9 - Threshold table: what each cutoff costs operationally
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9]:
    alert = y_scores >= t
    rows.append({
        "threshold": t,
        "alert_rate_%": round(alert.mean() * 100, 2),
        "precision": round(yt[alert].mean(), 3) if alert.any() else np.nan,
        "recall": round((yt & alert).sum() / yt.sum(), 3),
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# CELL 10 - Two-tier fraud policy (actually executed this time)
BLOCK_THRESHOLD = 0.75        # high confidence -> auto-block
INVESTIGATE_THRESHOLD = 0.40  # medium -> route to AI-assisted investigation
# ^ Revisit both using the cell-9 table after re-running.

block = y_scores >= BLOCK_THRESHOLD
review = (y_scores >= INVESTIGATE_THRESHOLD) & ~block
approve = ~block & ~review

print(f"auto-block:   {block.mean():7.2%} of txns | "
      f"precision {yt[block].mean():.3f} | {yt[block].sum():,} frauds caught")
print(f"investigate:  {review.mean():7.2%} of txns | "
      f"precision {yt[review].mean():.3f} | {yt[review].sum():,} frauds caught")
print(f"auto-approve: {approve.mean():7.2%} of txns")
print(f"fraud captured by block+investigate: "
      f"{yt[block | review].sum() / yt.sum():.1%}")

In [ ]:
# CELL 11 - Feature importance (explainability, saved)
xgb.plot_importance(model, max_num_features=14, importance_type="gain")
plt.title("Top Fraud Drivers (XGBoost, gain)")
plt.tight_layout()
plt.savefig("../reports/xgb_baseline_feature_importance.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# CELL 12 - Save model + metrics (dashboard/API can serve these)
import joblib, json

joblib.dump(model, "../models/xgb_fraud_model.pkl")

metrics = {
    "roc_auc": round(float(roc), 4),
    "pr_auc": round(float(ap), 4),
    "base_rate_test": round(float(base), 4),
    "split": "time-based 70/10/20 (train/val/test)",
    "features": features,
    "block_threshold": BLOCK_THRESHOLD,
    "investigate_threshold": INVESTIGATE_THRESHOLD,
}
with open("../models/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))